In [0]:
USE CATALOG esg_assurance;
USE SCHEMA adi_assurance;

In [0]:
CREATE OR REPLACE TABLE gold_scope3_category_summary AS

SELECT
    scope3_category,
    COUNT(*) AS transaction_count,
    COUNT(DISTINCT supplier_id) AS supplier_count,
    ROUND(SUM(spend_usd), 2) AS spend_usd,
    ROUND(SUM(calculated_emissions_tco2e), 6) AS emissions_tco2e,

    SUM(
        CASE
            WHEN calculation_method = 'supplier-specific'
            THEN 1 ELSE 0
        END
    ) AS supplier_specific_records,

    SUM(
        CASE
            WHEN calculation_method = 'spend-based'
            THEN 1 ELSE 0
        END
    ) AS spend_based_records

FROM silver_reporting_eligibility

WHERE reporting_eligibility = 'ELIGIBLE'

GROUP BY scope3_category;

In [0]:
SELECT *
FROM gold_scope3_category_summary
ORDER BY scope3_category;

In [0]:
SELECT
    reporting_eligibility,
    COUNT(*) AS transaction_count,
    ROUND(SUM(calculated_emissions_tco2e), 6) AS emissions_tco2e
FROM esg_assurance.adi_assurance.silver_reporting_eligibility
GROUP BY reporting_eligibility
ORDER BY reporting_eligibility;

In [0]:
SELECT
    e.control_id,
    e.exception_type,
    COUNT(DISTINCT e.transaction_id) AS affected_transactions,
    ROUND(
        SUM(DISTINCT r.calculated_emissions_tco2e),
        6
    ) AS affected_emissions_tco2e
FROM esg_assurance.adi_assurance.assurance_exception_log e
JOIN esg_assurance.adi_assurance.silver_reporting_eligibility r
    ON e.transaction_id = r.transaction_id
WHERE r.reporting_eligibility = 'HOLD'
  AND e.remediation_status <> 'CLOSED'
GROUP BY
    e.control_id,
    e.exception_type
ORDER BY
    e.control_id;

In [0]:
CREATE OR REPLACE TABLE
esg_assurance.adi_assurance.assurance_reconciliation AS

SELECT
    '01_BRONZE_SOURCE' AS stage,
    COUNT(*) AS record_count,
    ROUND(SUM(calculated_emissions_tco2e), 6) AS emissions_tco2e

FROM esg_assurance.adi_assurance.bronze_supplier_procurement

UNION ALL

SELECT
    '02_REMEDIATED_SILVER',
    COUNT(*),
    ROUND(SUM(calculated_emissions_tco2e), 6)

FROM esg_assurance.adi_assurance.silver_supplier_procurement_remediated

UNION ALL

SELECT
    '03_ELIGIBLE_FOR_REPORTING',
    COUNT(*),
    ROUND(SUM(calculated_emissions_tco2e), 6)

FROM esg_assurance.adi_assurance.silver_reporting_eligibility
WHERE reporting_eligibility = 'ELIGIBLE'

UNION ALL

SELECT
    '04_ON_HOLD',
    COUNT(*),
    ROUND(SUM(calculated_emissions_tco2e), 6)

FROM esg_assurance.adi_assurance.silver_reporting_eligibility
WHERE reporting_eligibility = 'HOLD'

UNION ALL

SELECT
    '05_GOLD_DISCLOSURE',
    SUM(transaction_count),
    ROUND(SUM(emissions_tco2e), 6)

FROM esg_assurance.adi_assurance.gold_scope3_category_summary;

In [0]:
SELECT *
FROM esg_assurance.adi_assurance.assurance_reconciliation
ORDER BY stage;

In [0]:
SELECT
    CASE
        WHEN ABS(
            (
                SELECT SUM(calculated_emissions_tco2e)
                FROM esg_assurance.adi_assurance.silver_reporting_eligibility
                WHERE reporting_eligibility = 'ELIGIBLE'
            )
            -
            (
                SELECT SUM(emissions_tco2e)
                FROM esg_assurance.adi_assurance.gold_scope3_category_summary
            )
        ) < 0.000001

        THEN 'PASS'
        ELSE 'FAIL'
    END AS source_to_disclosure_reconciliation;